<a href="https://colab.research.google.com/github/padminidokka/AI-Based-Dental-Radiograph-Analysis-System/blob/main/Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install gradio tensorflow opencv-python matplotlib reportlab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import cv2
import gradio as gr

In [ ]:
classes = [
    "Normal",
    "Cavity",
    "Implant",
    "Missing Tooth",
    "Impacted"
]

In [ ]:
model_path = "/content/drive/MyDrive/Dental/implant_classifier_fast.pth"

model = models.resnet18(weights=None)

model.fc = nn.Linear(model.fc.in_features, 5)

state_dict = torch.load(model_path, map_location="cpu")

model.load_state_dict(state_dict)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

In [ ]:
import tensorflow as tf

seg_model_path = "/content/drive/MyDrive/Dental/dental_model.h5"

seg_model = tf.keras.models.load_model(seg_model_path)

In [ ]:
IMG_SIZE = 256

def preprocess_seg(image):

    image = cv2.resize(image,(IMG_SIZE,IMG_SIZE))
    image = image / 255.0
    image = np.expand_dims(image,axis=0)

    return image

In [ ]:
def color_mask(mask):

    mask = np.argmax(mask, axis=-1)

    color = np.zeros((mask.shape[0], mask.shape[1],3), dtype=np.uint8)

    color[mask==1] = [0,255,0]     # green
    color[mask==2] = [255,0,0]     # red
    color[mask==3] = [0,0,255]     # blue
    color[mask==4] = [255,255,0]   # yellow

    return color

In [ ]:
def overlay_mask(image, mask):

    mask = cv2.resize(mask,(image.shape[1],image.shape[0]))

    overlay = cv2.addWeighted(image,0.7,mask,0.3,0)

    return overlay

In [ ]:
def predict_opg(image):

    if image is None:
        return None, "Upload image"

    image = np.array(image)

    # convert grayscale → RGB
    if len(image.shape) == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)

    # ---------- Segmentation ----------
    seg_input = preprocess_seg(image)

    seg_pred = seg_model.predict(seg_input)[0]

    mask = color_mask(seg_pred)

    overlay = overlay_mask(image, mask)

    # ---------- Classification ----------
    cls_input = transform(image).unsqueeze(0)

    with torch.no_grad():
        output = model(cls_input)

    pred = torch.argmax(output,1).item()

    diagnosis = classes[pred]

    return overlay, diagnosis

In [ ]:
interface = gr.Interface(
    fn=predict_opg,
    inputs=gr.Image(type="numpy", label="Upload OPG X-ray"),
    outputs=[
        gr.Image(label="Uploaded Image"),
        gr.Textbox(label="Diagnosis")
    ],
    title="Dental Radiograph Diagnostic Dashboard",
    description="Upload an OPG X-ray to detect dental conditions"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d68e92424a1c70cef2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
